# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library. All entities and fields are referenced by their Croissant `@id`, ensuring robust and reproducible code.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n")
print(f"Description: {metadata.description}\n")

## 2. Data Overview
Review all available record sets and their IDs.
We will gather their `@id`s so we can reference them in extraction steps.

In [ ]:
# List all available record sets in the dataset referenced by their @id
record_sets = list(dataset.record_sets)
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record set name: {rs['name']}, @id: {rs['@id']}")
    # Display available fields
    print("  Fields:")
    for field in rs.get('field', []):
        print(f"    {field['name']} (@id: {field['@id']})")
    print()

## 3. Data Extraction
Load all records from a specific record set using its `@id` into a pandas DataFrame for analysis. We will use one main clinical data record set as the example (referenced by its `@id`).

In [ ]:
# For this dataset, let's use the first tabular record set for demo.
main_record_set_id = record_sets[0]['@id']
print(f"Selected record set: {main_record_set_id}")

# List available record sets for extraction
record_set_ids = [rs['@id'] for rs in record_sets]

# Load all record sets into pandas DataFrames
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {len(df)} records from record set {rs_id}")
    except Exception as e:
        print(f"Could not load record set {rs_id}: {e}")

# Show columns of the main record set DataFrame
main_df = dataframes[main_record_set_id]
print(f"\nColumns in main DataFrame ({main_record_set_id}):\n{main_df.columns.tolist()}")
main_df.head()

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field by its `@id` and demonstrate basic filtering, normalization, and grouping. All field and column operations reference these `@id`s, not display names.

In [ ]:
# Select candidate numeric field with @id for demo (e.g., age at diagnosis, number of months between cancers, etc.)
# Please adjust 'age_at_second_crc' and 'sex' to match actual @id strings found above if needed.
numeric_field_id = None
group_field_id = None

# Print out columns to choose field IDs
print(f"Columns in main record set: {main_df.columns.tolist()}")

# Example field IDs (adjust as per your actual dataset):
for col in main_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
    if 'sex' in col.lower() or 'gender' in col.lower():
        group_field_id = col
if not numeric_field_id:
    # Default to first numeric column
    numeric_cols = main_df.select_dtypes('number').columns
    if len(numeric_cols) > 0:
        numeric_field_id = numeric_cols[0]
if not group_field_id:
    # Default to first object/categorical column
    group_cols = main_df.select_dtypes('object').columns
    if len(group_cols) > 0:
        group_field_id = group_cols[0]

# Print selected field IDs
print(f"Using numeric field: {numeric_field_id}")
print(f"Using group field: {group_field_id}")

try:
    threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 0
    filtered_df = main_df[main_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # If possible, group by the group field
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())
except Exception as e:
    print(f"EDA step failed: {e}")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, let's plot the normalized distribution of the chosen numeric field, and a breakdown by group.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style="whitegrid")

# Visualize the numeric field distribution
if numeric_field_id:
    fig, ax = plt.subplots(figsize=(7, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15, kde=True, ax=ax)
    ax.set_title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

# Visualize boxplot grouped by categorical group field, if available
if numeric_field_id and group_field_id:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=main_df[group_field_id], y=main_df[numeric_field_id])
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xlabel(group_field_id)
    plt.ylabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we loaded the FAIR² second primary colorectal cancer clinical dataset via its Croissant schema using the `mlcroissant` library, explored its record sets and fields by their `@id`, and demonstrated common analysis steps using only these standardized references. This ensures maximum reusability and reproducibility for research and model development. You can now further explore specific record sets or join with other Croissant datasets for richer biomedical insights.